## Specify the Knowledge Base to use

In [ ]:
from importlib.resources import files
import kmds
from kmds.tagging.tag_types import DataRepresentationTags
from owlready2 import *
from kmds.utils.load_utils import *
from kmds.utils.path_utils import get_package_kb_path
KNOWLEDGE_BASE = str(files("kmds.examples").joinpath("example_ml_kb_exp_workflow.xml"))
onto2 :Ontology = load_kb(KNOWLEDGE_BASE)
import pandas as pd
pd.options.display.max_colwidth = 500

## Load the Exploratory Observations

In [ ]:
load_exp_observations(onto2)

## Load the Data Representation Observations

In [ ]:
load_data_rep_observations(onto2)

## Load the Modelling Choice Observations

In [ ]:
load_modelling_choice_observations(onto2)

## Load the Model Selection Observations

In [ ]:
load_model_selection_observations(onto2)

In [ ]:
df_consolidated = load_observations(onto2)

In [ ]:
fp = str(files("kmds.examples").joinpath("example_ml_kb_summary.txt"))
df_consolidated.to_csv(fp, index=False)

## New Feature 1: Semantic Search Across Observations

This feature builds a semantic vector index from the project knowledge base and lets you ask natural-language queries.

What this gives you:
- You can discover related findings without exact keyword matches.
- You can retrieve cross-phase insights quickly from one query.

In [ ]:
from kmds.search import SemanticIndex

ml_idx = SemanticIndex()
ml_idx.build(KNOWLEDGE_BASE)

ml_query = "Which modelling decisions and evaluation outcomes were most important?"
ml_semantic_results = ml_idx.search(ml_query, n_results=5)

print(f"Query: {ml_query}")
for i, row in enumerate(ml_semantic_results, start=1):
    print(f"{i}. {row['obs_type']} | {row['finding']}")
    print(f"   distance={row['distance']:.4f}")

## New Feature 2: LLM Search Orchestrator (Template Routing + Fallback)

The orchestrator adds an LLM routing layer on top of search templates.

What this gives you:
- Intent routing: selects the best observation template for your question.
- Structured extraction: converts natural-language constraints into validated filter fields.
- Safe fallback: falls back to semantic search when routing is uncertain or returns no template results.

The cell below uses a local demo router function so this notebook runs without requiring an API key.

In [ ]:
import json
from kmds.search import SearchOrchestrator


def demo_llm_router_and_synth(prompt: str) -> str:
    # Router response in strict JSON schema expected by the orchestrator.
    if "Return JSON only" in prompt:
        route = {
            "intent_class": "model_selection_search",
            "filters": {
                "keyword": "evaluation",
                "finding_seq_min": 1
            },
            "explanation": "The query targets model-selection metrics and outcomes."
        }
        return json.dumps(route)

    # Synthesis response.
    return "Model-selection metrics favored the final model on validation outcomes and practical constraints."


ml_orchestrator = SearchOrchestrator(
    kb_path=KNOWLEDGE_BASE,
    llm_fn=demo_llm_router_and_synth,
    n_results=5,
)

ml_orchestrated = ml_orchestrator.ask(
    "Which evaluation metrics drove the final model choice?"
)

print("Intent class:", ml_orchestrated.intent_class)
print("Route explanation:", ml_orchestrated.route_explanation)
print("Answer:", ml_orchestrated.answer)
print("Returned records:", len(ml_orchestrated.results))

## New Feature 3: Natural Language Observation Ingestion

This feature lets you describe a finding in plain English and have KMDS classify it,
extract structured entities, and optionally log it to the knowledge base — without
writing manual ontology code.

Two modes are available inside a notebook:

- **Summary mode**: classify a text statement and get a human-readable description of the matching observation type.
- **Mapping mode**: get the full structured mapping object including the observation family, type, extracted entities, and validation status.

In [ ]:
from kmds.utils.natural_language_observation import summarize_observation_text, map_text_to_observation

# Summary mode: plain-text classification of a finding
text = "Customer ID and Description columns contain null values that need to be handled before model training."
print(summarize_observation_text(text))

In [ ]:
# Mapping mode: structured output — family, type, entities, validation
mapping = map_text_to_observation(text)
print(f"Family    : {mapping.workflow_family}")
print(f"Type      : {mapping.observation_type}")
print(f"Intent    : {mapping.intent}")
print(f"Component : {mapping.extracted_entities.affected_component}")
print(f"Valid     : {mapping.validation_passed}")

In [ ]:
# Log mode: convert natural language directly into a KMDS knowledge base entry
# Uncomment the block below to write a new observation to the knowledge base.

# from kmds.utils.natural_language_observation import log_text_as_observation
# result = log_text_as_observation(
#     text=text,
#     workflow_name="retail_customer_modelling",
#     project_file_path=KNOWLEDGE_BASE,
#     project_mode="update",
# )
# print(result.mapping.observation_type, "logged to", result.project_file)